# Illustrative recalibration of IE Carbon Budgets

In [ ]:
import sys
import types
import glob
import os
import re
import urllib.parse as urlp
import urllib.request as urlr
import shutil
import zipfile as zipf

#import calendar
#display(list(calendar.day_abbr))

#import pytz
import numpy as np

import matplotlib.pylab as plt
#import matplotlib.dates as mdates

import pandas as pd

In [ ]:
# display() variant that will show *all* rows and cols

# USE SPARINGLY!
# display_all() on large dataframes can cause large memory use and severe slow down on browser side!

def display_all(x) :
    with pd.option_context('display.max_rows', None,  'display.max_columns', None):
        display(x)

In [ ]:
# Adapted from: https://stackoverflow.com/questions/38358521/alternative-of-urllib-urlretrieve-in-python-3-5
def wget(url,file) :
    with urlr.urlopen(url) as response, open(file, 'wb') as out_file:
        shutil.copyfileobj(response, out_file)

## Load GWP100 factors (AR5 and AR6 variants)

+ via [Ireland's Final Greenhouse Gas Emissions 1990-2021](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2021.php), table on p.30. 
+ Table manual extracted to a `.ods` file.
+ HFCs and PFCs are not broken out individually, but shown with a range (for both AR4 and AR5). For other gases there is just a single value (shown in the "low" columns).
+ For the current "illustrative example" of recalibrating CBs we currently ignore AR4-5 adjustments for HFCs and PFCs: so strip out those gases (and now redundant "low-high" column labels).

In [ ]:
gwp100 = pd.read_excel("GWPs-AR4-AR5.ods", sheet_name="GWPs-AR4-AR5", header=0, index_col=1, thousands=',')
gwp100 = gwp100.drop(['CO2','HFCs','PFCs'], axis='index') 
  # CO2 GWP100 = 1.0 by definition; HFCs and PFCs not unbundled, so also not usable for current purposes 
gwp100 = gwp100.drop(columns=['Industrial designation or common name', 'AR4-high', 'AR5-high'])
gwp100 = gwp100.rename({'AR4-low':'AR4','AR5-low':'AR5'},axis='columns')
gwp100['AR4-5-rescale']  = gwp100['AR5']/gwp100['AR4']
display(gwp100)

## NIR/CRF data

We *have* to use the NIR data (rather than domestic "GHG Emissions" report) at least up to the 2022 cycle (1990-2020), because up to and including this point, EPA were not including LULUCF in the domestic report. It is a bit more tedious than the domestic report because the NIR CRF uses a separate file for each year, rather than the integrated single-file format of the domestic report. There again, it includes more detail, and is arguably the "authoritative" data.


### UNFCCC NIR page links (all parties) and Ireland IE CRF zipfile links

Manually harvested. 

+ [2026](https://unfccc.int/ghg-inventories-annex-i-parties/2026)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/IRL-CRT-2026-V1.0.zip)
+ [2025](https://unfccc.int/ghg-inventories-annex-i-parties/2025)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/IRL-CRT-2025-V1.2-20250314-111705_awaiting%20approval.zip)
+ [2024](https://unfccc.int/ghg-inventories-annex-i-parties/2024)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/IRL-CRT-2024-V1.0-1990-2022-20241217-102401.zip) 
+ [2023](https://unfccc.int/ghg-inventories-annex-i-parties/2023)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2023-crf-14apr23_AR5.zip) 
+ [2022](https://unfccc.int/ghg-inventories-annex-i-parties/2022)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2022-crf-7sep22.zip)
+ [2021](https://unfccc.int/ghg-inventories-annex-i-parties/2021)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2021-crf-13apr21.zip)
+ [2020](https://unfccc.int/ghg-inventories-annex-i-parties/2020)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2020-crf-15apr20.zip)
+ [2019](https://unfccc.int/process-and-meetings/transparency-and-reporting/reporting-and-review/reporting-and-review-under-the-convention/greenhouse-gas-inventories-annex-i-parties/submissions/national-inventory-submissions-2019)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2019-crf-10apr19.zip)
+ [2018](https://unfccc.int/process-and-meetings/transparency-and-reporting/reporting-and-review/reporting-and-review-under-the-convention/greenhouse-gas-inventories-annex-i-parties/submissions/national-inventory-submissions-2018)
    + [IE CRF zip](https://unfccc.int/sites/default/files/resource/irl-2018-crf-6nov18.zip)
+ [2017](https://unfccc.int/process-and-meetings/transparency-and-reporting/reporting-and-review/reporting-and-review-under-the-convention/greenhouse-gas-inventories-annex-i-parties/submissions/national-inventory-submissions-2017)
    + [IE CRF zip](https://unfccc.int/files/national_reports/annex_i_ghg_inventories/national_inventories_submissions/application/zip/irl-2017-crf-12apr17.zip)


In [ ]:
# Manually harvested
unfccc_crf_urls = [
    'https://unfccc.int/sites/default/files/resource/IRL-CRT-2026-V1.0.zip',
    'https://unfccc.int/sites/default/files/resource/IRL-CRT-2025-V1.2-20250314-111705_awaiting%20approval.zip',
    'https://unfccc.int/sites/default/files/resource/IRL-CRT-2024-V1.0-1990-2022-20241217-102401.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2023-crf-14apr23_AR5.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2022-crf-7sep22.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2021-crf-13apr21.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2020-crf-15apr20.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2019-crf-10apr19.zip',
    'https://unfccc.int/sites/default/files/resource/irl-2018-crf-6nov18.zip',
    'https://unfccc.int/files/national_reports/annex_i_ghg_inventories/national_inventories_submissions/application/zip/irl-2017-crf-12apr17.zip']

unfccc_crf_zipfiles = pd.DataFrame(columns = ['url','zipfile','GWP100s'])
for u in unfccc_crf_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    zipfile = url_path_split[-1]
    year = int(re.search(r'[^0-9]*([0-9]{4})', zipfile).group(1))
    if (year >= 2023):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'
    unfccc_crf_zipfiles.loc[year] = [u,zipfile,gwp100s]

In [ ]:
display(unfccc_crf_zipfiles)

#### But ... UNFCCC CRF links not usable in python?

Sadly, while the CRF links work fine within a (firefox) browser, they don't work via python - not clear what the issue is, but presumably something to do with blocking robot scraping etc. Urg.

There are some indications of attempts at an API for UNFCCC GHG data, but nothing that seems currently stable/working.

+ [UNFCCC: GHG Data Interface](https://ghgdi.unfccc.int/)
+ [zenodo: UNFCCC country-submitted greenhouse gas emissions data until 2024-07-04](https://zenodo.org/records/12657469)
+ [unfccc-di-api 4.0.4](https://pypi.org/project/unfccc-di-api/)


### epa.ie NIR landing page and CRF zipfile links [Ireland only]

Manually harvested. Can't find anything earlier than 2019 - doh!

+ [2026](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2026.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL-CRT-2026-V1.0.zip)
+ [2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2025.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland-Common-Reporting-Tables-(CRT)-2025.zip)
+ [2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2024.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2024__12032024_130024_started.zip)
+ [2023](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2023.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2023_14042023_160711.zip) 
+ [2022](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2022.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2022_06092022_174936_website.zip)
+ [2021](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2021.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2021__10032021_160609_started.zip)
+ [2020](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2020.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2020_2015_06032020_151848_started-(1).zip)
+ [2019](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-national-inventory-submissions-2019.php)
    + [IE CRF zip](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Submission-files_CRF.zip)


In [ ]:
# Manually harvested
epa_crf_urls = [
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL-CRT-2026-V1.0.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland-Common-Reporting-Tables-(CRT)-2025.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2024__12032024_130024_started.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2023_14042023_160711.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2022_06092022_174936_website.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2021__10032021_160609_started.zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/IRL_2020_2015_06032020_151848_started-(1).zip',
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Submission-files_CRF.zip']

crf_root = 'indata/crf'
epa_crf_zipfiles = pd.DataFrame(columns = ['url','folder','zipfile','zippath','GWP100s','skiprows','index_col'])
for u in epa_crf_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    zipfile = url_path_split[-1]
    try :
        year = int(re.search(r'[^0-9]*([0-9]{4})', zipfile).group(1))
    except : 
        year = 2019 # VERY UGLY AND FRAGILE HACK!
    if (year >= 2023):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'
    if (year >= 2025) :
        skiprows = 7
        index_col = 1
    else :
        skiprows = 4
        index_col = 0
    folder = crf_root+'/'+str(year)
    zippath = folder+'/'+zipfile
    epa_crf_zipfiles.loc[year] = [u,folder,zipfile,zippath,gwp100s,skiprows,index_col]

In [ ]:
display(epa_crf_zipfiles)

### Download and unzip NIR/CRF files

In [ ]:
def get_unzip_crf(year) :
    src = epa_crf_zipfiles.loc[year,:]

    if (os.path.exists(src["folder"])) :
        print(F'crf folder already exists: {src["folder"]}')
    else :
        os.mkdir(src['folder'])

    if(os.path.exists(src['zippath'])) :
        print(F'crf zip file already exists: {src["zippath"]}')
    else :
        wget(src['url'],src['zippath'])

    with zipf.ZipFile(src['zippath'], 'r') as zip_ref:
        zip_ref.extractall(src['folder'])
    

In [ ]:
for year in epa_crf_zipfiles.index :
    print(year)
    get_unzip_crf(year)

### Load and preprocess NIR-CRF data

Up to and including the 2022 cycle (covering 1990-2020), the EPA were still using AR4 GWP100 values for aggregation. Given that the CCAC already "manually" updated for AR5 values even on the first cycle of the CB programme, we need to already make that adjustment here, where applicable.

In [ ]:
def load_single_invfile(invfile, skiprows, index_col, gwp100s) :

    raw_totals = pd.read_excel(invfile, 
        sheet_name="Summary2", 
        skiprows=skiprows,
        index_col=index_col,
        nrows=2, header=0).transpose()

    # Drop empty leading columns (if any); transposed to rows here...
    raw_totals = raw_totals.iloc[index_col :, :]

    # Flatten minor(?) variations in row and column names across CRF years
    #print(raw_totals[year].columns)
    for colname in raw_totals.columns :
        if re.match(".*Total.*net emissions.*",colname) :
            raw_totals = raw_totals.rename(columns={colname:'Total (net emissions)'})

    for rowname in raw_totals.index :
        if re.match("CO2",rowname) :
            raw_totals = raw_totals.rename(index={rowname:'CO2'})

    raw_totals = raw_totals.drop('Unspecified mix of HFCs and PFCs',axis='index') # All "NO"
    
    return(raw_totals)

In [ ]:
def parse_crf_fileyear(crf_file) :
    path_split = crf_file.split('/')
    filename = path_split[-1]

    try :
      inventory_year = int(re.search('^.*[_-][\d]{4}.*[_-]([\d]{4})[^\d].*xlsx$', filename).group(1))
    except Exception as err:
        print(f"Unexpected {err=}, {type(err)=}")
        inventory_year = 0
    #print(F'filename: {filename} inventory_year: {inventory_year}')
    return (inventory_year)


In [ ]:
def load_single_crf(crf_zipfiles, crfyear) :
    crf_invyear_filepattern = crf_root+'/'+str(crfyear)+"/IRL*.xlsx"

    crf_invyear_files = pd.DataFrame(glob.glob(crf_invyear_filepattern), columns=['file'])
        # glob to avoid picking up extraneous files, such as lock files if some files open in calc application!
    
    crf_invyears = crf_invyear_files['file'].apply(parse_crf_fileyear)
    
    crf_invyears.name = 'invyear'
    crf_invyear_files.index = crf_invyears
    crf_invyear_files = crf_invyear_files.sort_index()
    crf_invyear_files = crf_invyear_files.loc[2015:,:] # Retrict to CBref years and beyond

    raw_totals = {}
    for invyear in crf_invyear_files.index :
        #print(F'invyear: {invyear}')
        raw_totals[invyear] = load_single_invfile(
            invfile=crf_invyear_files.at[invyear,'file'],
            skiprows=crf_zipfiles.loc[crfyear,'skiprows'], 
            index_col=crf_zipfiles.loc[crfyear,'index_col'], 
            gwp100s=crf_zipfiles.loc[crfyear,'GWP100s'])
    
    crf_df = pd.DataFrame()
    invyear0 = list(raw_totals.keys())[0] # Assume indices are the same for all invyears in any single crfyear!
    crf_df.index = raw_totals[invyear0].index

    for invyear in crf_invyear_files.index :
        crf_df[invyear] = raw_totals[invyear]['Total (net emissions)']
    
    crf_df.loc['Total-xchk'] = crf_df['CO2':'NF3'].sum(axis='index')
    
    return crf_df

In [ ]:
def load_crf_data(crf_zipfiles) :

    crf_data = {} 
    for crf_year in crf_zipfiles.index :
        print(F'crf_year: {crf_year}')
        crf_data[crf_year] = load_single_crf(crf_zipfiles,crf_year)

    return(crf_data)

In [ ]:
crf_data_raw = load_crf_data(epa_crf_zipfiles)

In [ ]:
for crf_year in crf_data_raw.keys() :
    print(F'crf_year: {crf_year}')
    display(crf_data_raw[crf_year])

In [ ]:
def rescale_ar4_ar5(inv_data) :
    for gas in gwp100.index :
        rescale = gwp100.loc[gas, 'AR4-5-rescale']
        inv_data.loc[gas,:] *= rescale

    return(inv_data)

In [ ]:
crf_data_ar5 = {}
for crf_year in crf_data_raw.keys() :
    crf_data_ar5[crf_year] = crf_data_raw[crf_year].copy()
    
    if (epa_crf_zipfiles.at[crf_year,'GWP100s'] == 'AR4') :
        crf_data_ar5[crf_year] = rescale_ar4_ar5(crf_data_ar5[crf_year])

    crf_data_ar5[crf_year].loc['Total CO2e (AR5)',:] = crf_data_ar5[crf_year].loc['CO2':'NF3'].sum(axis='index')

In [ ]:
for crf_year in crf_data_ar5.keys() :
    print(F"crf_year: {crf_year} GWP100s: {epa_crf_zipfiles.at[crf_year,'GWP100s']}")
    display(crf_data_ar5[crf_year])

## EPA "Provisional" GHG Inventory data

### Load and preprocess EPA "Provisional" GHG Inventory data

Note that provisional inventories seem to have used an AR5 GWP100 basis since the 2022 cycle (1990-2021); and that the provisional inventories prior to that excluded LULUCF. Thus ... for CB purposes, provisional inventories prior to 2022 are not useful (no LULUCF), so all provisional inventories of interest here are already using AR5 GWP100s and don't need correction. The code included below does demonstrate how AR4-5 GWP100 correction *could* be done if necessary (albeit only for CH4 and N2O); but it is not actually applicable in practice.


### epa.ie Provisional Inventory landing page and datafile links [Ireland only]

Manually harvested.

+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2025](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2025.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2025_web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2024.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2024_web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2023.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2023_web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2022.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions-1990-2022_Web.xlsx) 
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2021.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_AR5_Web.xlsx)
    (File name states "final" and "AR5", suggesting that it might actually be the 2023 "final" emissions report; but file properties show date of "21/07/2022, 11:49:44" so it *is* the 2022 "provisional" report, and
    the data does actually differ from the actual 2023 "final" report. Doh!)
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2020.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions_1990-2020_web.xlsx) (exc. LULUCF)
+ [Ireland's Provisional Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-provisional-greenhouse-gas-emissions-1990-2019.php)
    + [IE pinv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_emissions_1990-2019_web.xlsx)  (exc. LULUCF)


In [ ]:
# Manually harvested
epa_pinv_urls = [
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2025_web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2024_web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_Emissions_1990-2023_web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions-1990-2022_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_AR5_Web.xlsx' 
    # Prior to this, files don't include LULUCF - doh!
    #'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional-emissions_1990-2020_web.xlsx', 
    #'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Provisional_emissions_1990-2019_web.xlsx'
    ]

In [ ]:
pinv_root = 'indata/pinv'
epa_pinv_xlsxfiles = pd.DataFrame(columns = ['url','folder','xlsxfile','xlsxpath','GWP100s','ystart','yend'])
for u in epa_pinv_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    xlsxfile = url_path_split[-1]
    try :
        match = re.search(r'[^\d]*([\d]{4})[-_]([0-9]{4})', xlsxfile)
        pinv_year_start = int(match.group(1))
        pinv_year_end = int(match.group(2))
        pinv_year_published = pinv_year_end + 1 # heuristic!!
    except Exception as err:
        print(f"Unexpected {err=}, {type(err)=}")
        pinv_year_start = pinv_year_end = pinv_year_published = 0
    if (pinv_year_published >= 2022):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'
    
    folder = pinv_root+'/'+str(pinv_year_published)
    xlsxpath = folder+'/'+xlsxfile
    epa_pinv_xlsxfiles.loc[pinv_year_published] = [u,folder,xlsxfile,xlsxpath,gwp100s,pinv_year_start,pinv_year_end]

In [ ]:
display(epa_pinv_xlsxfiles)

### Download Provisional Inventory files

In [ ]:
def get_pinv_xlsx(pinv_year) :
    src = epa_pinv_xlsxfiles.loc[pinv_year,:]

    if (os.path.exists(src["folder"])) :
        print(F'pinv folder already exists: {src["folder"]}')
    else :
        os.mkdir(src['folder'])

    if(os.path.exists(src['xlsxpath'])) :
        print(F'pinv xlsx file already exists: {src["xlsxpath"]}')
    else :
        wget(src['url'],src['xlsxpath'])


In [ ]:
for pinv_year in epa_pinv_xlsxfiles.index :
    print(pinv_year)
    get_pinv_xlsx(pinv_year)

### Load and preprocess Provisional Inventory data

Up to and including the 2021 provisional inventory cycle (covering 1990-2020), the EPA were still using AR4 GWP100 values for aggregation. Given that the CCAC already "manually" updated for AR5 values even on the first cycle of the CB programme, we *would* need to already make that adjustment here, if applicable. The code provides for that ... but in practice it isn't actually used (because all provisional inventories that include LULUCF are already using AR5 GWP100s).

In [ ]:
def load_single_pinv_xlsx(invfile, gwp100s, year_start, year_end) :

    period = F'{year_start:4d}-{year_end:4d}'
    raw_sheets = pd.read_excel(invfile, 
        sheet_name=None, # load multiple sheets; returns dict of sheet names -> df 
        skiprows=0,
        index_col=0,
        header=0)

    raw_totals = pd.DataFrame(columns=range(year_start,year_end+1))
    for sheet_name in raw_sheets.keys() :
        if (not(re.match('.*ETS|.*CAP',sheet_name))) :
            raw_totals.loc[sheet_name,:] = raw_sheets[sheet_name].loc['National Total with LULUCF',year_start:year_end]            

    # Flatten row names?
    row_name_map = {}
    for row in raw_totals.index :
        if (re.match('.*GHG',row)) :
            row_name_map[row] = 'Total CO2e'
        elif (re.match('.*CO2',row)) :
            row_name_map[row] = 'CO2'
        elif (re.match('.*CH4',row)) :
            row_name_map[row] = 'CH4'
        elif (re.match('.*N2O|.*N20',row)) :
            row_name_map[row] = 'N2O'
    raw_totals = raw_totals.rename(index=row_name_map)
    raw_totals = raw_totals.loc[:,2015:]

    raw_totals.loc['Other GHG'] = raw_totals.loc['Total CO2e']-raw_totals.loc['CO2':'N2O',:].sum(axis='index')
    
    return(raw_totals)

In [ ]:
def load_pinv_data(pinv_xlsxfiles) :

    pinv_data = {} 
    for pinv_year in pinv_xlsxfiles.index :
        print(F'pinv_year: {pinv_year}')
        invfile = pinv_xlsxfiles.at[pinv_year, 'xlsxpath']
        gwp100s = pinv_xlsxfiles.at[pinv_year, 'GWP100s']
        year_start = pinv_xlsxfiles.at[pinv_year, 'ystart']
        year_end = pinv_xlsxfiles.at[pinv_year, 'yend']
        pinv_data[pinv_year] = load_single_pinv_xlsx(invfile=invfile,
            gwp100s=gwp100s, year_start=year_start, year_end=year_end)

    return(pinv_data)

In [ ]:
pinv_data_raw = load_pinv_data(epa_pinv_xlsxfiles)

In [ ]:
for pinv_year in pinv_data_raw.keys() :
    print(F'pinv_year: {pinv_year}')
    display(pinv_data_raw[pinv_year])

### "Consistently-inconsistent" treatment of LULUCF in most recent inventory year?

There is an odd issue that `Other GHG` is sometimes showing up as negative; but only in the "most recent" inventory year, and only for publications years of 2025 and 2026. Looking
at the spreadsheet, the discrepancy seems to be relating to LULUCF: on the `GHG` sheet these numbers are simply copied from the previous year; but on the `CO2`, `CH4` and `N2O` sheets 
they seem to have distinct ("real"?) data for the year in question. So that would naturally cause an issue with the totals inc. LULUCF. Ho hum.

### Rescale GWP100s AR4-AR5

This code demonstrates the principle; but since, in practice, all the provisional inventories in scope actually already use AR5 GWP100s, it's a bit moot.

In [ ]:
def pinv_rescale_ar4_ar5(inv_data) :
    for gas in gwp100.index :
        rescale = gwp100.loc[gas, 'AR4-5-rescale']
        inv_data.loc[gas,:] *= rescale
    return(inv_data)

In [ ]:
pinv_data_ar5 = {}
for pinv_year in pinv_data_raw.keys() :
    pinv_data_ar5[pinv_year] = pinv_data_raw[pinv_year].copy()
    
    if (epa_pinv_xlsxfiles.at[pinv_year,'GWP100s'] == 'AR4') :
        pinv_data_ar5[pinv_year] = rescale_ar4_ar5(pinv_data_ar5[pinv_year])

    pinv_data_ar5[pinv_year].loc['Total CO2e (AR5)',:] = pinv_data_ar5[pinv_year].loc['CO2':'Other GHG'].sum(axis='index')

In [ ]:
for pinv_year in pinv_data_ar5.keys() :
    print(F"pinv_year: {pinv_year} GWP100s: {epa_pinv_xlsxfiles.at[pinv_year,'GWP100s']}")
    display(pinv_data_ar5[pinv_year])

## EPA "Final" GHG Inventory data

### Load and preprocess EPA "Final" GHG Inventory data

In general the NIR/CRF data is preferable to the domestic "Final Greenhouse Gas emissions" reporting from the EPA. They should ultimately agree (exactly?) on the totals, the CRF provides more granular detail, should it be needed. The one circumstance that might arise (?) would be if the Final GHG Report comes out a little earlier than the NIR submission, though that should be a fairly narrow window?

Nonetheless, for completeness, we include code here for fetching and loading EPA Final GHG datafiles.

### epa.ie Final Inventory landing page and datafile links [Ireland only]

Manually harvested.

+ [Ireland's Final Greenhouse Gas Emissions 1990-2024](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2024.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2024_Web.xlsx) 
+ [Ireland's Final Greenhouse Gas Emissions 1990-2023](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2023.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2023_Web.xlsx) 
+ [Ireland's Final Greenhouse Gas Emissions 1990-2022](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2022.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2022_Web.xlsx) 
+ [Ireland's Final Greenhouse Gas Emissions 1990-2021](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-final-greenhouse-gas-emissions-1990-2021.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_Webv2.xlsx)
+ [Ireland's Final Greenhouse Gas Emissions 1990-2020](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/greenhouse-gas-emissions-final-2020.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2020_AR4_web.xlsx) (exc. LULUCF)
+ [Ireland's Final Greenhouse Gas Emissions 1990-2019](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/greenhouse-gas-emissions-final-2019.php)
    + [IE finv xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions_1990-2019_web.xlsx) (exc. LULUCF)
 



In [ ]:
# Manually harvested
epa_finv_urls = [
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2024_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG-Final-emissions-data-1990-2023_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2022_Web.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2021_Webv2.xlsx'
    # Prior to this, files don't include LULUCF - doh!
    #'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions-data_1990-2020_AR4_web.xlsx',
    #'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/GHG_Final-emissions_1990-2019_web.xlsx'
    ]

finv_root = 'indata/finv'
epa_finv_xlsxfiles = pd.DataFrame(columns = ['url','folder','xlsxfile','xlsxpath','GWP100s','ystart','yend'])
for u in epa_finv_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    xlsxfile = url_path_split[-1]
    try :
        match = re.search(r'[^\d]*([\d]{4})[-_]([0-9]{4})', xlsxfile)
        finv_year_start = int(match.group(1))
        finv_year_end = int(match.group(2))
        finv_year_published = finv_year_end + 2 # heuristic!!
        #finv_year = int(re.search(r'[^\d]*[\d]{4}[-_]([0-9]{4})', xlsxfile).group(1)) + 2
    except Exception as err:
        print(f"Unexpected {err=}, {type(err)=}")
        finv_year_start = finv_year_end = finv_year_published = 0
    if (finv_year_published >= 2023):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'

    folder = finv_root+'/'+str(finv_year_published)
    xlsxpath = folder+'/'+xlsxfile
    epa_finv_xlsxfiles.loc[finv_year_published] = [u,folder,xlsxfile,xlsxpath,gwp100s,finv_year_start,finv_year_end]


In [ ]:
display(epa_finv_xlsxfiles)

### Download Final Inventory files

**FIXME:** merge get_pinv_xlsx() and get_finv_xlsx()?

In [ ]:
def get_finv_xlsx(inv_year) :
    src = epa_finv_xlsxfiles.loc[inv_year,:]

    if (os.path.exists(src["folder"])) :
        print(F'finv folder already exists: {src["folder"]}')
    else :
        os.mkdir(src['folder'])

    if(os.path.exists(src['xlsxpath'])) :
        print(F'finv xlsx file already exists: {src["xlsxpath"]}')
    else :
        wget(src['url'],src['xlsxpath'])


In [ ]:
for finv_year in epa_finv_xlsxfiles.index :
    print(finv_year)
    get_finv_xlsx(finv_year)

### Load and preprocess Final Inventory data

Up to and including the 2022 final inventory cycle (covering 1990-2020), the EPA were still using AR4 GWP100 values for aggregation. Given that the CCAC already "manually" updated for AR5 values even on the first cycle of the CB programme, we *would* need to already make that adjustment here, if applicable. The code provides for that ... but in practice it isn't actually used (because all final inventories that include LULUCF are already using AR5 GWP100s).

**FIXME:** merge load_single_pinv_xlsx() and load_single_finv_xlsx()?

In [ ]:
def load_single_finv_xlsx(invfile, gwp100s, year_start, year_end) :

    period = F'{year_start:4d}-{year_end:4d}'
    raw_sheets = pd.read_excel(invfile, 
        sheet_name=None, # load multiple sheets; returns dict of sheet names -> df 
        skiprows=0,
        index_col=0,
        header=0)

    raw_totals = pd.DataFrame(columns=range(year_start,year_end+1))
    for sheet_name in raw_sheets.keys() :
        if (not(re.match('.*ETS|.*CAP',sheet_name))) :
            raw_totals.loc[sheet_name,:] = raw_sheets[sheet_name].loc['National Total with LULUCF',year_start:year_end]            

    # Flatten row names?
    row_name_map = {}
    for row in raw_totals.index :
        if (re.match('.*GHG',row)) :
            row_name_map[row] = 'Total CO2e'
        elif (re.match('.*CO2',row)) :
            row_name_map[row] = 'CO2'
        elif (re.match('.*CH4',row)) :
            row_name_map[row] = 'CH4'
        elif (re.match('.*N2O|.*N20',row)) :
            row_name_map[row] = 'N2O'
    raw_totals = raw_totals.rename(index=row_name_map)
    raw_totals = raw_totals.loc[:,2015:]

    raw_totals.loc['Other GHG'] = raw_totals.loc['Total CO2e']-raw_totals.loc['CO2':'N2O',:].sum(axis='index')
    
    return(raw_totals)

In [ ]:
def load_finv_data(finv_xlsxfiles) :

    finv_data = {} 
    for finv_year in finv_xlsxfiles.index :
        print(F'finv_year: {finv_year}')
        invfile = finv_xlsxfiles.at[finv_year, 'xlsxpath']
        gwp100s = finv_xlsxfiles.at[finv_year, 'GWP100s']
        year_start = finv_xlsxfiles.at[finv_year, 'ystart']
        year_end = finv_xlsxfiles.at[finv_year, 'yend']
        finv_data[finv_year] = load_single_finv_xlsx(invfile=invfile,
            gwp100s=gwp100s, year_start=year_start, year_end=year_end)

    return(finv_data)

In [ ]:
finv_data_raw = load_finv_data(epa_finv_xlsxfiles)

In [ ]:
for finv_year in finv_data_raw.keys() :
    print(F'finv_year: {finv_year}')
    display(finv_data_raw[finv_year])

### Rescale GWP100s AR4-AR5 (dummy!)

As there are no files in scope here (i.e., inc. LULUCF) that use pre-AR5 GWP100s, we just copy `finv_raw_data` but renaming `Total CO2e` to `Total CO2e (AR5)` (for consistency with approach to CRF and provisional inventories above).

In [ ]:
finv_data_ar5 = {}
for finv_year in finv_data_raw.keys() :
    finv_data_ar5[finv_year] = finv_data_raw[finv_year].rename(index={'Total CO2e' : 'Total CO2e (AR5)'})


In [ ]:
display(finv_data_ar5)

## EPA GHG Projections data

### Load and preprocess EPA GHG Projections data


### epa.ie Projections landing page and datafile links [Ireland only]

Manually harvested.

+ [Ireland's Greenhouse Gas Emissions Projections 2025-2030](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-greenhouse-gas-emissions-projections-2025-2030.php)
    > "The focus of the assessment
    > in this report is out to 2030 given current EU and National climate targets, and that many existing policy
    > instruments do not extend beyond 2030. Projections data out to 2055 are provided as a separate
    > download accompanying this report."
    + [IE ghgprj xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Irelands_2026_GHG_Emission_Projections_2025-2055_incLULUCF.xlsx) (2025-2055)
+ [Ireland's Greenhouse Gas Emissions Projections 2024-2055](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-greenhouse-gas-emissions-projections-2024-2055.php)
    + [IE ghgprj xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Irelands_2025_GHG_Emission_Projections_2024-2055_incLULUCF.xlsx) 
+ [Ireland's Greenhouse Gas Emissions Projections 2023-2050](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-greenhouse-gas-emissions-projections-2023-2050.php)
    + [IE ghgprj xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Irelands_2024_GHG_Emission_Projections_2023-2050_incLULUCF.xlsx) 
+ [Ireland's Greenhouse Gas Emissions Projections 2022-2040](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-greenhouse-gas-emissions-projections-2022-2040.php)
    + [IE ghgprj xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland_2023_GHG_Emission_Projections_2022-2040_incLULUCF.xlsx) 
+ [Ireland's Greenhouse Gas Emissions Projections 2021-2040](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-greenhouse-gas-emissions-projections-2021-2040.php)
    > "The Climate Action and Low Carbon Development (Amendment) Act 2021 sets a national climate objective
    > to achieve a reduction of 51% in total emissions (including from Land Use, Land Use Change and Forestry
    > - LULUCF) over the period 2018 to 2030. This is the first set of national projections prepared by the EPA
    > following the enactment of this legislation and since the Climate Action Plan 2021 was published in
    > November 2021."
    This report says they switched to AR5 GWP100s (p. 14). *However*: the data file *still* doesn't include LULUCF!? Doh!
    + [IE ghgprj xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Data-file-for-Web---Ireland_2022_GHG_Emission_Projections_2021-2040v1.xlsx) 
+ [Ireland's Greenhouse Gas Emissions Projections 2020-2040](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-greenhouse-gas-emissions-projections-2020-2040.php)
    + [IE ghgprj xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland_2021_GHG_Emission_Projections_2020-2040.xlsx) 
+ [Ireland's Greenhouse Gas Emissions Projections 2019-2040](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/irelands-greenhouse-gas-emissions-projections-2019-2040.php)
    + [IE ghgprj xlsx](https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland_2020_GHG_Emission_Projections_2019-2040.xlsx) 


In [ ]:
# Manually harvested
epa_ghgprj_urls = [
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Irelands_2026_GHG_Emission_Projections_2025-2055_incLULUCF.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Irelands_2025_GHG_Emission_Projections_2024-2055_incLULUCF.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Irelands_2024_GHG_Emission_Projections_2023-2050_incLULUCF.xlsx', 
    'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland_2023_GHG_Emission_Projections_2022-2040_incLULUCF.xlsx', 
    # Prior to this, files don't include LULUCF - doh!
    #'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Data-file-for-Web---Ireland_2022_GHG_Emission_Projections_2021-2040v1.xlsx', 
    #'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland_2021_GHG_Emission_Projections_2020-2040.xlsx', 
    #'https://www.epa.ie/publications/monitoring--assessment/climate-change/air-emissions/Ireland_2020_GHG_Emission_Projections_2019-2040.xlsx', 
    ]

ghgprj_root = 'indata/ghgprj'
epa_ghgprj_xlsxfiles = pd.DataFrame(columns = ['url','folder','xlsxfile','xlsxpath','GWP100s','ystart','yend'])
for u in epa_ghgprj_urls :
    url_split = urlp.urlsplit(u)
    url_path_split = url_split.path.split('/')
    xlsxfile = url_path_split[-1]

    ghgprj_year_published = ghgprj_year_start = ghgprj_year_end = 0 # Default placeholder
    try :
        match = re.search(r'[^\d]*([\d]{4})[^\d]*([\d]{4})[-_]([\d]{4})', xlsxfile)
        ghgprj_year_published = int(match.group(1))
        ghgprj_year_start = int(match.group(2))
        ghgprj_year_end = int(match.group(3))
    except Exception as err:
        print(f"Unexpected {err=}, {type(err)=}")

    if (ghgprj_year_published >= 2022):
        gwp100s = 'AR5'
    else :
        gwp100s = 'AR4'

    folder = ghgprj_root+'/'+str(ghgprj_year_published)
    xlsxpath = folder+'/'+xlsxfile
    epa_ghgprj_xlsxfiles.loc[ghgprj_year_published] = [u,folder,xlsxfile,xlsxpath,gwp100s,ghgprj_year_start,ghgprj_year_end]


In [ ]:
display(epa_ghgprj_xlsxfiles)

### Download GHG Projections files

**FIXME:** merge get_ghgprj_xlsx(), get_finv_xlsx(), get_pinv_xlsx()?

In [ ]:
def get_ghgprj_xlsx(ghgprj_published) :
    src = epa_ghgprj_xlsxfiles.loc[ghgprj_published,:]

    exist_ok=True
    if(os.path.exists(src['xlsxpath'])) :
        print(F'ghgprj xlsx file already exists: {src["xlsxpath"]}')
    else :
        os.makedirs(src["folder"], exist_ok=True)
        wget(src['url'],src['xlsxpath'])
        

In [ ]:
for ghgprj_year in epa_ghgprj_xlsxfiles.index :
    print(f'{ghgprj_year} ({epa_ghgprj_xlsxfiles.at[ghgprj_year,"ystart"]}-{epa_ghgprj_xlsxfiles.at[ghgprj_year,"yend"]})')
    get_ghgprj_xlsx(ghgprj_year)

### Load and preprocess GHG Projections data

Up to and including the 2021 GHG projections cycle (covering 2020-2040), the EPA were still using AR4 GWP100 values for aggregation. Given that the CCAC already "manually" updated for AR5 values even on the first cycle of the CB programme, we *would* need to already make that adjustment here, if applicable. However, up to and including the 2022 GHG projections cycle (covering 2021-2040), the EPA still excluded LULUCF from the data file. So all projections files that include LULUCF are already using AR5 GWP100s, so no AR4-AR5 adjustments are required.

In [ ]:
def load_single_ghgprj_xlsx(ghgprj_file, gwp100s, year_start, year_end) :

    period = F'{year_start:4d}-{year_end:4d}'
    raw_sheets = pd.read_excel(ghgprj_file, 
        sheet_name=None, # load multiple sheets; returns dict of sheet names -> df 
        skiprows=0,
        index_col=0,
        header=0)

    raw_totals = pd.DataFrame(columns=range(year_start,year_end+1))
    for sheet_name in raw_sheets.keys() :
        if (not(re.match('.*ETS',sheet_name))) :
            if (sheet_name == 'With Existing Measures') :
                raw_totals.loc['Total CO2e (WEM)',:] = raw_sheets[sheet_name].loc['National Total including LULUCF',year_start:year_end]            
            elif (sheet_name == 'With Additional Measures') :
                raw_totals.loc['Total CO2e (WAM)',:] = raw_sheets[sheet_name].loc['National Total including LULUCF',year_start:year_end]
            else :
                print(f'Unexpected sheet_name: {sheet_name}!?')

    raw_totals = raw_totals.loc[:,:2039] # de facto limit of current analysis (CB programme cycle 2)
    
    return(raw_totals)

In [ ]:
def load_ghgprj_data(ghgprj_xlsxfiles) :

    ghgprj_data = {} 
    for ghgprj_year in ghgprj_xlsxfiles.index :
        print(F'ghgprj_year: {ghgprj_year}')
        ghgprj_file = ghgprj_xlsxfiles.at[ghgprj_year, 'xlsxpath']
        gwp100s = ghgprj_xlsxfiles.at[ghgprj_year, 'GWP100s']
        year_start = ghgprj_xlsxfiles.at[ghgprj_year, 'ystart']
        year_end = ghgprj_xlsxfiles.at[ghgprj_year, 'yend']
        ghgprj_data[ghgprj_year] = load_single_ghgprj_xlsx(ghgprj_file=ghgprj_file,
            gwp100s=gwp100s, year_start=year_start, year_end=year_end)

    return(ghgprj_data)

In [ ]:
ghgprj_data_raw = load_ghgprj_data(epa_ghgprj_xlsxfiles)

In [ ]:
for ghgprj_year in ghgprj_data_raw.keys() :
    print(F'ghgprj_year: {ghgprj_year}')
    display(ghgprj_data_raw[ghgprj_year])

### Rescale GWP100s AR4-AR5 (dummy!)

As there are no files in scope here (i.e., inc. LULUCF) that use pre-AR5 GWP100s, we just have a dummy assignment (for consistency with approach to CRF and inventories above).

In [ ]:
ghgprj_data_ar5 = ghgprj_data_raw

## Rescale CBs and project outcomes based on inventory (crf/pinv/finv) and projections data

We run the numbers for a variety of reference inventory and reporting year index cases. 


In [ ]:
def cb_refinv_label(case) :
    return(f"{case['inv_data']}_{case['refinv_start']}-{case['refinv_duration']}Y")

cb_refinv_cases = pd.DataFrame(columns=['inv_data', 'refinv_start', 'refinv_duration'],
    data = [
     ['crf',2018,1],
     ['finv',2018,1],
     ['pinv',2018,1],
     ['crf',2016,5],
     ['finv',2016,5],
     ['pinv',2016,5],
     ['crf',2015,5],
     ['finv',2015,5],
     ['pinv',2015,5]])
cb_refinv_cases['refinv_end'] = cb_refinv_cases['refinv_start'] + cb_refinv_cases['refinv_duration'] - 1 
cb_refinv_cases.index = cb_refinv_cases.apply(cb_refinv_label,axis='columns',)

assert(cb_refinv_cases.index.is_unique) # Defensive!
display(cb_refinv_cases)


In [ ]:
def summarise_inv_data(inv_data, refinv_start, refinv_end) :
    summary = pd.Series()
    for reporting_year in inv_data.keys() :
        inv_years = inv_data[reporting_year].columns
        if ((refinv_start in inv_years) and
            (refinv_end in inv_years)) :
                summary[reporting_year] = (
                    inv_data[reporting_year].loc['Total CO2e (AR5)',refinv_start:refinv_end].sum()/1000.0)
    summary = summary.sort_index(ascending=True)
    return(summary)

In [ ]:
inv_reporting_years = sorted(list(set(crf_data_ar5.keys()) | set(pinv_data_ar5.keys()) | set(finv_data_ar5.keys())))

display(inv_reporting_years)

In [ ]:
inv_data_map = {'crf' : crf_data_ar5, 'pinv' : pinv_data_ar5, 'finv' : finv_data_ar5}

cb_refinv_data = pd.DataFrame(columns=inv_reporting_years)
for case in cb_refinv_cases.index :
    print(f"refinv case: {case}")
    inv_data = inv_data_map[cb_refinv_cases.at[case,'inv_data']]
    year_start = cb_refinv_cases.at[case,'refinv_start']
    year_end = cb_refinv_cases.at[case,'refinv_end']
    refinv_data_single = summarise_inv_data(inv_data, year_start, year_end)
    cb_refinv_data.loc[case,refinv_data_single.index] = refinv_data_single

In [ ]:
display(cb_refinv_data)

In [ ]:
cb_refinv_index_years = [2021,2022]
cb_refinv_index_cases = pd.DataFrame(columns=['refinv_case', 'index_year'])

for index_year in cb_refinv_index_years :
    for refinv_case in cb_refinv_cases.index :
        refinv_index_case_label = F"{refinv_case}@{index_year}"
        cb_refinv_index_cases.loc[refinv_index_case_label,:] = [refinv_case,index_year]

In [ ]:
display(cb_refinv_index_cases)

In [ ]:
cb_refinv_index_data = pd.DataFrame(columns=cb_refinv_data.columns)

for refinv_index_case in cb_refinv_index_cases.index :
    index_year = cb_refinv_index_cases.at[refinv_index_case,'index_year']
    refinv_case = cb_refinv_index_cases.at[refinv_index_case,'refinv_case']
    index_inv = cb_refinv_data.at[refinv_case,index_year]
    if (not np.isnan(index_inv)) :
        cb_refinv_index_data.loc[refinv_index_case,:] = cb_refinv_data.loc[refinv_case, :]/index_inv

In [ ]:
display(cb_refinv_index_data)

#### "Shortlisting" cases of interest?

Note specifically that, per the [CCAC CB Cycle 1 Technical Report](https://tinyurl.com/CCAC-CB-TR)
and (more importantly?) the "liberated" CCAC CB Cycle 1 [Ireland Scenarios with LULUCF 2021 Cleanish [Excel workbook]](https://zenodo.org/records/7075669), their
analysis and modelling relied on the 2021 NIR/CRF, with AR4-5 adjustment. These is corroborated by their specific citing of 2018 total emissions (inc LULUCF) as 68.3MtCO2e (AR5), derived 
from reported 67.3MtCO2e (AR4); those totals specifically match the totals here in `crf_data_ar5[2021]` and `crf_data_raw[2021]` respectively.

So, a "shortlist" of candicate reference inventory cases that "match" the basis for the original cycle 1 modelling would be:

+ `crf_2018-1Y@2021`
+ `crf_2016-5Y@2021`
+ `crf_2015-5Y@2021`

However: `crf_2016-5Y@2021` (or "CB0" as I have previously termed it) must then drop out, because it only provides inventory data up to 2019; so we are left with the other two ("statutory" 1Y of 2018 
and "latest available" 5Y of 2015-2019). The relevant 5Y (2021-2026) scaling factors then barely differ:  

In [ ]:
display(cb_refinv_index_data.loc[['crf_2018-1Y@2021', 'crf_2015-5Y@2021'],2026])

ratio_5Y_1Y = cb_refinv_index_data.at['crf_2015-5Y@2021',2026]/cb_refinv_index_data.loc['crf_2018-1Y@2021',2026]
print(F'ratio_5Y_1Y: {ratio_5Y_1Y}')

So just c. 0.6% in the difference - not material.

Now I am still inclined to argue for the use of the 5Y reference; and the format of using the "latest available" 5Y period in *whatever* inventory fed into the relevant CCAC modelling (i.e., on the basis 
of whenever the budget programme "under review" was proposed). This is both because it gives a more "comparable" statistical reference for the 5Y periods making up the CB programme, and because
the *narrow* role of 2018 in the Act (and in the original CB process) has now been discharged. But ... there remains an argument that it is "relevant" up to the review cycle when the CB2 outturn is reported, 
and "carry forward" dealt with (so 2032!?). But *after* that (next review due in 2037) the 1Y 2018 reference will *certainly* be fully obsoleted, and it will make sense to shift to a 5Y comparator?

Though ... for the *immediate* question, since there is no material difference in the `crf_2018-1Y@2021` and `crf_2015-5Y@2021` rescaling factors, it is not worth making this a major issue for this first iteration
of rescaling.

Further complication: the concept of the CCAC modelling from 'whenever the budget programme "under review" was proposed' is itself problematic, because in general, a CB programme may span elements from two
cycles of budgetting (the "overlapping", "provisional" third budget in each cycle).

So ... there's actually a fair question as to whether the "best" point in the cycle for undertaking the re-scaling is at the point of carry forward, or the point of proposing the "next" cycle. This is 
difference that makes a difference: on the "current" cycle, the roll forward of the CB programme recommendations was undertaken in 2024 (statutory deadline was end 2024), presumably with the 2024 NIR/CRF 
data as an input (would need to check that?). Whereas, the "carry forward" update is not due until 2027 (presumably when the 2027 NIR/CRF data is published). Worse ... the programme roll forward is
*still* in abeyance (not approved): so there is a real question about how to deal with it in the context of re-scaling. If it *had* been completed, then CB3 would have already been "revised" based on a 
2024 CRF; so it would - presumably - not make sense to rescale it based on indexing 2018 emissions over 2021-2025; rather it should "only" reflect the indexing over 2021-2024.

Urg!!



### Projected CB outcomes (analysis as of November 2026)

This just summarises projected CB1-3 outcomes based on a particular combination of inventory and projections data, and allowing for carry forward of budget exceedance or surplus.

We have a provisional inventory for 2026 covering 2021-2025 (albeit with "placeholder" LULUCF for 2025) and projections for 2025-2055 (of which we will use 2026-2035).

+ Recalculate CB1-CB3 for each case in cb_refinv_index_data 
    + Questionmark around whether to expand case variants for `index_duration`, or `index_end_year` as it were; but for now, just use 2026 as single case option?
+ Calculate projected outcomes for each cb_refinv case for both WEM and WAM projections
    + Not expected to be hugely different; but let's just have the numbers to see.

In [ ]:
cb_cyc1 = {'CB1' : 295.0, 'CB2': 200.0, 'CB3': 151.0}
cb_cyc1 = pd.Series(data=cb_cyc1)
#display(cb_cyc1)

rescale_reporting_year = 2026
cb_rescale_cases = pd.DataFrame(columns=['scale_factor'])
cb_rescale_cases.at['No-rescale','scale_factor'] = 1.0

cb_rescale_cases = pd.concat([cb_rescale_cases,cb_refinv_index_data[rescale_reporting_year].rename('scale_factor')])

for cb in cb_cyc1.keys() :
    cb_rescale_cases.loc[:,cb] = cb_rescale_cases.loc[:,'scale_factor']*cb_cyc1[cb]

In [ ]:
display(cb_rescale_cases)

In [ ]:
def cb_process_overshoot(cb) :
    cb.loc['CB1','adjusted'] = cb.loc['CB1','cyc1']
    cb.loc['CB1','overshoot'] = cb.loc['CB1','projection']-cb.loc['CB1','adjusted']
    
    cb.loc['CB2','adjusted'] = cb.loc['CB2','cyc1']-cb.loc['CB1','overshoot']
    cb.loc['CB2','overshoot'] = cb.loc['CB2','projection']-cb.loc['CB2','adjusted']
    
    cb.loc['CB3','adjusted'] = cb.loc['CB3','cyc1']-cb.loc['CB2','overshoot']
    cb.loc['CB3','overshoot'] = cb.loc['CB3','projection']-cb.loc['CB3','adjusted']

    cb['overshoot (%)'] = (cb['overshoot']/cb['adjusted']) * 100.
    return(cb)

In [ ]:
cb_projections = {}

for prj_case in ['WEM','WAM'] :
    for cb_rescale_case in cb_rescale_cases.index :
        prj_case_label = f"{cb_rescale_case}-{prj_case}x{cb_rescale_cases.loc[cb_rescale_case,'scale_factor']:5.3f}"
        #print(F'case: {prj_case_label}')
        cb = pd.DataFrame(index = ['CB1','CB2','CB3'])

        cb['cyc1'] = cb_rescale_cases.loc[cb_rescale_case,'CB1':'CB3']
        
        cb.at['CB1','projection'] = pinv_data_ar5[2026].loc['Total CO2e (AR5)',2021:2025].sum()/1000.
        cb.at['CB2','projection'] = ghgprj_data_ar5[2026].loc[F'Total CO2e ({prj_case})',2026:2030].sum()/1000.
        cb.at['CB3','projection'] = ghgprj_data_ar5[2026].loc[F'Total CO2e ({prj_case})',2031:2035].sum()/1000.

        cb = cb_process_overshoot(cb)
        #display(cb)

        cb_projections[prj_case_label] = cb
        

In [ ]:
for prj_case in cb_projections.keys() :
    print(F'case: {prj_case}')
    display(cb_projections[prj_case])

## LEGACY/EXPERIMENTAL CODE - to be removed...

### Extract/summarise CRF data for "statutory" reference inventory year (2018)

### Extract/summarise CRF data for "notional" CB cycle 1 reference period (2015-2019)

Roughly speaking (!) this is the latest 5 year period for which a CRF was available prior to the CCAC finalising their cycle 1 proposals.

### Extract/summarise CRF data for CB0 period

## Preliminary experimentation with crf files/data

## Preliminary experimentation with finv files/data

In [ ]:
infile = "EPA-Inv-Final-1990-2020/GHG_Final-emissions-data_1990-2020_AR4_web.xlsx"
epa_final_1990_2020 = pd.DataFrame()
epa_final_1990_2020 = pd.read_excel(infile, sheet_name="NEW Summary 1990-2020 GHG", header=0, index_col=0)

In [ ]:
display_all(epa_final_1990_2020)

In [ ]:
epa_final_1990_2020_cb0 = epa_final_1990_2020.loc['National Total with LULUCF',2016:2020]

In [ ]:
display(epa_final_1990_2021_cb0)